# 论文基本信息

| 项目        | 内容                                                                                               |
| :-------- | :----------------------------------------------------------------------------------------------- |
| **标题**    | Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model? |
| **作者**    | Yang Yue, Zhiqi Chen, Rui Lu, Andrew Zhao, Zhaokai Wang, Shiji Song, Gao Huang                   |
| **机构**    | LeapLab, Tsinghua University; Shanghai Jiao Tong University                                      |
| **arXiv** | [2504.13837]https://arxiv.org/abs/2504.13837                                                  |
| **发表时间**  | 2025年4月（NeurIPS 2025）                                                                            |
| **项目主页**  | [limit-of-RLVR.github.io](https://limit-of-RLVR.github.io)                                       |

# 研究背景与核心问题
## 当前共识
RLVR（Reinforcement Learning with Verifiable Rewards）被认为是推动 LLM 推理能力跃迁的核心技术。从 OpenAI o1 到 DeepSeek-R1，业界普遍认为 RLVR 能让模型：
  - 自主探索新的推理策略
  - 自我改进，获得超越基础模型的新能力
  - 涌现出反思、枚举、迭代优化等高级行为
  - "It is widely believed that, similar to how traditional RL helps agents explore and learn new strategies, RLVR enables LLMs to continuously self-improve, thus acquiring novel reasoning abilities that exceed the capacity of the corresponding base models." 
  
## 论文的质疑
作者提出了一个根本性的反问：<font color='red'>RLVR 真的让模型学会了基础模型不会的东西吗？</font>

传统评估指标（如 pass@1、平均准确率）只衡量"一次采样答对的概率"，无法区分两种截然不同的机制：

| 机制            | 含义                               |
| :------------ | :------------------------------- |
| **A. 发现新能力**  | RLVR 让模型掌握了基础模型完全不会的推理路径         |
| **B. 提升采样效率** | RLVR 只是让基础模型**本来就有的**正确路径更容易被采样到 |

论文的核心假设是：<font color='red'>当前 RLVR 可能只是机制 B，而非机制 A</font>

# 核心方法：Pass@k 作为能力边界探针

##  Pass@k？

Pass@k 的定义：**给模型 k 次独立采样机会，只要有一次正确就算通过**。

| k 值                 | 衡量的东西                       |
| :------------------ | :-------------------------- |
| **k = 1 (Pass\@1)** | 单次采样效率——模型"第一反应"有多准         |
| **k 很大 (如 256)**    | **能力边界**——<font color='red'>模型**理论上**能解决的问题集合 </font>|


### 关键洞察：
  - "What if we invest heavily in sampling for the base model with an enormous k? Can its performance match the RLVR-trained one?" 
  
### 如果 RLVR 真的赋予了模型新的推理能力，那么：
  - **RLVR 模型在 大 k 时应该解决更多问题（因为它会了基础模型不会的）**
  - Base 模型即使采样再多，也追不上
  
### 如果 RLVR 只是提升效率：
  - RLVR 模型在 k=1 时表现更好（更擅长一次命中）
  - 但 Base 模型在 大 k 时可能追平甚至超越（因为它本来就会，只是需要多试几次）

# 四大核心发现
## 发现 1：Base 模型在大 k 时 超过 RLVR 模型
<font color='red'>这是论文最反直觉的发现：</font>
 -  "base models consistently surpass RL-trained models across all benchmarks and LLM families as k increases. More notably, at sufficiently large k, base models achieve even higher pass@k scores than their RL-enhanced counterparts." 
  
### 具体表现：
  - Math 任务（AIME、MATH500 等）：Base 模型在 k=256 时 pass@k 高于 RLVR 模型
  - Code 任务（LiveCodeBench）：同样趋势
  - Visual Reasoning：跨模态任务中也观察到
  
### 这意味着：
  - <font color='blue'>RLVR 模型能解决的问题，Base 模型通过充分采样也能解决</font> 
  - 而 Base 模型能解决的一些问题，RLVR 模型反而解决不了了。
  
## 发现 2：<font color='red'>RLVR 提升效率，但缩小能力边界</font>

论文用一张图清晰展示了这一 trade-off：

In [ ]:
Pass@k
  ↑
  │    RLVR (k=1) ████          Base (k=256) ████████
  │    RLVR (k=256) ███               ↑
  │                              Base 覆盖更广！
  │    ───────────────────────────────
  │         k=1        k=256
  └────────────────────────────────────→ k

"RLVR training contributes to reasoning by biasing the model toward rewarded reasoning paths, thereby increasing the likelihood of sampling correct reasoning paths. However, this efficiency gain also comes at a cost: RL training reduces the model's exploration capacity, resulting in smaller coverage of solvable problems at large k." 

### 机制解释：
- <font color='red'>RLVR 通过强化"高奖励路径"，让模型更倾向于走这些路。</font>
  - **但这也导致模型过度特化，探索范围变窄——原本基础模型偶尔能走通的"非主流"正确路径，被 RLVR "修剪"掉了。**


## 发现 3：<font color='red'>所有推理路径已存在于 Base 模型中</font>
### 论文通过困惑度分析（Perplexity Analysis）验证：
  - "reasoning paths generated by RLVR-trained models already exist within the base models' output distribution with considerable probability density, indicating that these reasoning patterns and CoTs are not totally strange and unachievable for base models." 
  
### 换句话说：
  - RLVR 模型生成的正确 CoT，在 Base 模型的采样分布中本来就有非零概率
  - <font color='red'>RLVR 没有创造新的推理模式，只是放大了已有模式中的正确分支</font>

## 发现 4：六种 RLVR 算法表现相近，且远未达最优
论文测试了六种主流 RLVR 算法：
- PPO
- GRPO
- REINFORCE++
- DAPO
- RLOO
- 其他变体
  - "different RL algorithms (e.g., PPO, GRPO, Reinforce++) show slight variations in performance, it does not make essential difference...the gap remains consistently large across all methods. This suggests that current RL approaches, which primarily work by improve sampling efficiency, are still far from optimal." 
  
作者定义了一个指标 <font color='red'>ΔSE （Sampling Efficiency Gap）= RL 模型的 pass@1 与 Base 模型的 pass@k（k=256，作为潜力上限）之间的差距。</font>
  - 所有 RL 算法的 ΔSE  都很大，说明现有方法都没有充分利用 Base 模型的推理潜力。

# RLVR vs Distillation：本质不同
论文对比了 RLVR 和知识蒸馏（Distillation），发现两者有本质区别：

|                   | RLVR                | Distillation             |
| :---------------- | :------------------ | :----------------------- |
| **机制**            | <font color='red'>从模型自身采样中筛选高奖励路径 </font>    | 从教师模型学习新的推理模式            |
| **能力边界**          | <font color='red'>**不扩展**，受限于 Base 模型</font> | <font color='red'>**可以扩展**，引入 Base 模型没有的知识 </font>|
| **Pass\@k (大 k)** | 可能下降                | 通常提升                     |
| **本质**            | <font color='red'>"挖掘已有潜力"  </font>          | "注入新知识"                  |

- "distillation can introduce new reasoning patterns from the teacher and genuinely expand the model's reasoning capabilities. In contrast, RLVR-trained models whose capacity remains bounded by the base." 

这是一个深刻的区分：
- 如果你想让模型学会它本来不会的东西，蒸馏可能更有效；
- RLVR 只是让模型更擅长用它已有的东西。

# 实验设置

## 模型家族
- Qwen2.5-Math（0.5B, 1.5B, 7B）
- Qwen3-8B
- Llama3.2-3B
- DeepSeek 系列

## 任务

| 领域   | 数据集                                        |
| :--- | :----------------------------------------- |
| 数学推理 | AIME24, MATH500, AMC, GSM8K, OlympiadBench |
| 代码生成 | LiveCodeBench                              |
| 视觉推理 | 多模态推理基准                                    |

## 训练设置
- Zero-RL：<font color='red'>直接从 Base 模型开始 RLVR，无 SFT 中间步骤（数学任务）</font>
- Instruction-tuned 起点：代码和视觉任务（遵循社区惯例）

# 对现有认知的挑战
这篇论文挑战了几个广为流传的观点：

| 流行观点                 | 论文的发现                           |
| :------------------- | :------------------------------ |
| "RLVR 让模型自我进化，获得新能力" | RLVR 只是**筛选和放大**已有能力，**不创造**新能力 |
| "R1 的反思行为是 RL 教出来的"  | 反思行为已存在于 Base 模型中，RL 只是让它更频繁出现  |
| "不同 RL 算法有本质差异"      | <font color='red'>在"是否扩展能力边界"这个维度上，**所有主流算法表现相似** </font>|
| "RL 是通往 AGI 的关键"     | 当前 RLVR 范式**不足以**推动推理能力的根本性突破   |

# 局限与未来方向
## 局限
  - **主要关注可验证奖励任务（数学、代码）**，<font color='red'>对开放式推理（如创意写作、常识推理）的结论可能不同</font>
  - 大 k 采样计算成本极高，实验规模受限
  - 未深入分析多轮交互、工具使用等更复杂的 RL 场景
  
## 未来方向
论文明确指出了改进 RLVR 的可能路径：
  - Continual Scaling：持续扩展训练计算量，而非固定预算后停止
  - Multi-turn Agent-Environment Interaction：让模型与环境多轮交互，而非单轮生成-奖励
  - ** richer feedback**：超越二元奖励，引入过程奖励、自然语言反馈等更丰富的信号
  - 探索机制：设计能真正扩展能力边界的探索策略，而非只在已有分布内优化
  
# 一句话总结
- 这篇论文的核心结论是：
  - **当前 RLVR 并非"授人以渔"（教模型新的推理方法），而是"优中选优"**（让模型更频繁地采样它本来就会的正确路径）。
  - <font color='red'>它提升了效率，但牺牲了探索广度</font>
    - Base 模型通过充分采样能解决的问题，比 RLVR 模型更多。要真正实现推理能力的突破，需要超越现有 RLVR 范式的新方法。